# Preprocessing the MUDI Drug–Drug Interaction Dataset

This notebook preprocesses the MUDI (Multi-label Drug Interaction) dataset to prepare it for machine learning models that predict drug–drug interaction (DDI) types based on molecular structure.

Key Objectives

- Map drug identifiers to molecular structures (SMILES)

- Convert raw dataset into a structure-based representation

- Encode interaction types into numerical labels

- Prepare clean train and test datasets

- Create a combined dataset for downstream modeling

https://zenodo.org/records/15544552

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 22.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from rdkit import Chem
from rdkit.Chem import PandasTools

In [ ]:
data_path = '/content/drive/MyDrive/FYP/IRP/Data/MUDI'
train_df = pd.read_csv(os.path.join(data_path, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'dataset/test.csv'))


In [ ]:
train_df['Interaction'].value_counts()

,count
Interaction,
Synergism,186268
Antagonism,27320
New Adverse,7527


In [ ]:
len(train_df['Drug1'].unique())

989

In [ ]:
import json

# Load the JSON file
with open('/content/drive/MyDrive/FYP/IRP/Data/MUDI/drug_info.json', 'r') as f:
    drug_info = json.load(f)

# Number of unique drug IDs (keys of the dictionary)
num_unique_drugs = len(drug_info)
print(f"Number of unique drug IDs in drug_info.json: {num_unique_drugs}")

Number of unique drug IDs in drug_info.json: 1295


## Create a Mapping from Drug ID to SMILES

The drug IDs in the CSV files have the format "Compound::DBxxxxx", while the keys in drug_info are simply "DBxxxxx". Need to strip the prefix.

In [ ]:
# Function to clean drug ID
def clean_drug_id(drug_id):
    # Remove 'Compound::' prefix if present
    return drug_id.replace('Compound::', '')

# Build a dictionary that maps cleaned ID to SMILES
id_to_smiles = {}
for drug_id, info in drug_info.items():
    if 'smiles' in info:
        id_to_smiles[drug_id] = info['smiles']

print("Example mapping:")
print("DB00842 ->", id_to_smiles.get('DB00842', 'Not found'))

Example mapping:
DB00842 -> C1=CC=C(C=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O


## Map SMILES to Each Row

Create new columns smiles_a and smiles_b by applying the cleaning and mapping.

In [ ]:
def get_smiles(row, drug_col):
    raw_id = row[drug_col]
    clean_id = clean_drug_id(raw_id)
    return id_to_smiles.get(clean_id, None)

# Apply to both dataframes
for df in [train_df, test_df]:
    df['smiles_a'] = df.apply(lambda row: get_smiles(row, 'Drug1'), axis=1)
    df['smiles_b'] = df.apply(lambda row: get_smiles(row, 'Drug2'), axis=1)

# Check for missing SMILES (should be none if all drugs are in drug_info)
print("Missing SMILES in train:", train_df['smiles_a'].isna().sum() + train_df['smiles_b'].isna().sum())
print("Missing SMILES in test:", test_df['smiles_a'].isna().sum() + test_df['smiles_b'].isna().sum())

Missing SMILES in train: 0
Missing SMILES in test: 0


## Encode Interaction Labels


The dataset contains three interaction types:

- Synergism

- Antagonism

- New Effect (New Adverse)

Will map them to numeric values for classification.

In [ ]:
# Define mapping
interaction_map = {
    'Synergism': 0,
    'Antagonism': 1,
    'New Adverse': 2
}

# Apply mapping
train_df['label'] = train_df['Interaction'].map(interaction_map)
test_df['label'] = test_df['Interaction'].map(interaction_map)

# Verify distribution
print("Train label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

Train label distribution:
 label
0    186268
1     27320
2      7527
Name: count, dtype: int64

Test label distribution:
 label
0    71001
1    11914
2     6502
Name: count, dtype: int64


In [ ]:
# Select and save train data
train_processed = train_df[['smiles_a', 'smiles_b', 'label']].copy()
train_processed.to_csv(os.path.join(data_path, 'train_processed.csv'), index=False)

# Select and save test data
test_processed = test_df[['smiles_a', 'smiles_b', 'label']].copy()
test_processed.to_csv(os.path.join(data_path, 'test_processed.csv'), index=False)

print("Processed files saved:")
print("- train_processed.csv")
print("- test_processed.csv")

Processed files saved:
- train_processed.csv
- test_processed.csv


In [ ]:
train_processed.sample(5)

,smiles_a,smiles_b,label
194915,CC1=C(C(C(=C(N1)C#N)C(=O)OC)C2=CC(=CC=C2)[N+](...,CC(C)(C)[C@@H](C(=O)N[C@@H](CC1=CC=CC=C1)[C@H]...,0
171302,C[C@]12C[C@@H]([C@]3([C@H]([C@@H]1C[C@H]([C@@]...,CC[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@]2(C#C)O)CCC...,0
11500,CCOC(=O)C1=CN=CN1[C@H](C)C2=CC=CC=C2,C1C2C3=CC=CC=C3CC4=CC=CC=C4N2C(=N1)N,0
105419,CC1=CC(=C(C(=C1CC2=NCCN2)C)O)C(C)(C)C,CC(C)(C)NCC(C1=NC(=C(C=C1)O)CO)O,0
153820,CCCC(=O)NC1=CC(=C(C=C1)OCC(CNC(C)C)O)C(=O)C,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N2C...,0


In [ ]:
print(f"Number of duplicate rows in train_df: {train_df.duplicated().sum()}")
print(f"Number of duplicate rows in test_df: {test_df.duplicated().sum()}")

Number of duplicate rows in train_df: 0
Number of duplicate rows in test_df: 0


In [ ]:
train_processed.shape, test_processed.shape

((221115, 3), (89417, 3))

## Merge train + test

In [ ]:
full_dataset = pd.concat([train_processed, test_processed], axis=0, ignore_index=True)

# save to CSV
# full_dataset.to_csv(os.path.join(data_path, 'MUDI_full_dataset_processed.csv'), index=False)

print(f"Full dataset shape: {full_dataset.shape}")
print(f"Label distribution:\n{full_dataset['label'].value_counts()}")

Full dataset shape: (310532, 3)
Label distribution:
label
0    257269
1     39234
2     14029
Name: count, dtype: int64


In [ ]:
print(f"Number of duplicate rows in full dataset: {full_dataset.duplicated().sum()}")

Number of duplicate rows in full dataset: 0


In [ ]:
full_dataset.sample(5)

,smiles_a,smiles_b,label
296895,CC(=O)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CCC4...,CCCC1=NC(=C(N1CC2=CC=C(C=C2)C3=CC=CC=C3C4=NNN=...,2
127473,CCCCC(=O)N(CC1=CC=C(C=C1)C2=CC=CC=C2C3=NNN=N3)...,CC(CC1=CC=CC=C1)N,1
195730,C1CC[C@H]2CN(C[C@H]2C1)C(=O)C[C@H](CC3=CC=CC=C...,CCC1NC2=CC(=C(C=C2C(=O)N1)S(=O)(=O)N)Cl,1
207871,CCN1C=C(C(=O)C2=C1C=C(C=C2)C3=CC=NC=C3)C(=O)O,C[C@@H]1CN(CC[C@@]1(C2=CC=CC=C2)C(=O)O)C3CCC(C...,0
153225,C1=CC=C(C=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O,CC1=C(N=CN1)CSCCNC(=NC)NC#N,0


In [ ]:
full_dataset.shape

(310532, 3)